# 1단계: 소비자 데이터 구조 파악

**목표**: 분석을 시작하기 전에 `소비자.csv`가 어떻게 생겼는지 확인한다.

확인할 질문
1. 행은 몇 개이고, 고객은 몇 명인가? (한 행 = 고객 한 명인가?)
2. `customer_seq`는 무엇을 뜻하는가?
3. 결측값과 `-` 값은 어디에, 얼마나 있는가?
4. `product_period`는 기간인가, 연도인가?
5. Training과 Validation 데이터는 어떤 관계인가?

> 원본 zip은 압축을 풀지 않고 바로 읽는다. 파일 전체를 화면에 출력하지 않고, 집계 결과만 본다.

## 데이터 구성 메모: 각 데이터가 무엇인가

| 데이터 | 비유 | 실제 내용 | 출처 |
|---|---|---|---|
| `소비자.csv` (`TL_2.소비자.zip`) | **고객 DB** | 고객 335,506명의 속성과 보유 상품. 한 행 = 고객 1명 × 보유 상품 1개 (총 385,019행). 실제 고객이 아닌 **합성데이터** | 데이터설명서 「데이터 규모」 표 주석: "소비자 수는 335,506명이나 실제 데이터 행(row)은 385,019건이며, 이는 소비자 1명이 복수 상품을 보유하는 합성데이터의 특성을 반영" / 데이터설명서 「구축 내용」: "금융사(증권·보험) Data Mart 추출 후 합성·검증이 완료된 소비자 속성 데이터" |
| CoT 데이터 (`TL_1.CoT_*.zip`) | **상황별 전화영업 스크립트** | 고객 질문(`question`) → 추론 1~3단계(`cot1`~`cot3`) → 상품 추천(`answer`, `product_names`). 고객 연령·성별·질문유형별 상담 시나리오 | AI Hub 데이터 페이지 「데이터 구성」·「어노테이션 포맷」 / 데이터설명서 「가공데이터 세부 속성」 |
| 상품 JSON (`TS_1.상품_*.zip`) | (상품 카탈로그) | 증권 6종·보험 7종 상품 설명서, 상품당 JSON 1개 (10,000개) | 데이터설명서 「데이터 규모」 표 |

**문서 출처**
- 「30.데이터설명서.hwp」, 「30.활용 가이드라인.hwp」 (AI Hub 「금융상품·서비스 및 소비자 특성 데이터」, 한국지능정보사회진흥원)
- AI Hub 데이터 페이지의 데이터 구성·어노테이션 포맷 설명

**두 데이터의 관계 (직접 확인한 결과, 2026-11-10)**
- CoT에는 `customer_id`가 없다. 소비자.csv와 겹치는 항목은 `gender`, `age`뿐이라 고객 단위로는 연결되지 않는다.
- CoT는 Training 8,000건 / Validation 1,001건으로 실제 분할되어 있다. 소비자.csv와 상품 JSON은 폴더마다 같은 파일이다 (데이터설명서: "Training, Validation, Test 각 폴더에 동일한 1개 CSV 파일 업로드").
- CoT의 `product_names` 중 상품 JSON의 `product_name`과 정확히 일치하는 이름은 76%이다.

**확인 항목: CoT 스크립트 파일 안에 customer_id나 다른 연결 키가 있는지 확인** → 완료 (2026-11-10)
- CoT 파일 9,001개(Train 8,000 + Valid 1,001)가 모두 같은 키 12개로 구성되어 있다. `customer_id`나 고객 순번 같은 키는 없다.
- 파일 본문과 파일명 어디에도 `customer_id` 형식(64자리 16진수) 문자열이 없다.
- 파일명은 `보험_03_21122.json`처럼 `업종_질문유형번호_cot_id` 형태이다.
- 소비자.csv와 겹치는 항목은 `gender`, `age`뿐이다. 따라서 고객 단위로 연결할 키는 없다.


In [ ]:
# 필요한 라이브러리 불러오기
import zipfile
from pathlib import Path

import pandas as pd

# 표가 잘리지 않도록 출력 옵션 조정
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# 프로젝트 루트(finance-eda/) 기준 경로 (이 노트북은 notebooks/ 안에 있으므로 한 단계 위가 루트)
# 원본은 data/raw/ 아래에 있다 (D11)
BASE = Path.cwd().parent
TRAIN_ZIP = BASE / "data" / "raw" / "Training" / "02.라벨링데이터" / "TL_2.소비자.zip"
VALID_ZIP = BASE / "data" / "raw" / "Validation" / "02.라벨링데이터" / "VL_2.소비자.zip"
print(TRAIN_ZIP.exists(), VALID_ZIP.exists())

## 1-1. zip 안에 무엇이 들어 있나

In [ ]:
# zip 안의 파일 이름과 크기만 확인 (압축은 풀지 않음)
# 한글 파일명이 깨지지 않도록 cp949 인코딩으로 이름을 읽는다
for zp in [TRAIN_ZIP, VALID_ZIP]:
    with zipfile.ZipFile(zp, metadata_encoding="cp949") as z:
        for info in z.infolist():
            print(zp.parent.parent.name, "|", info.filename,
                  f"| 압축 {info.compress_size/1e6:.1f}MB → 원본 {info.file_size/1e6:.1f}MB")

## 1-2. 앞부분 몇 줄만 읽어 보기

In [ ]:
# zip 안의 CSV를 여는 함수: 매번 같은 코드를 쓰지 않도록 묶어 둔다
def open_csv(zp):
    z = zipfile.ZipFile(zp, metadata_encoding="cp949")
    name = z.namelist()[0]          # zip 안에 CSV가 하나뿐이므로 첫 번째 파일
    return z.open(name)

# 앞 5행만 읽기 (utf-8-sig: 파일 맨 앞의 BOM 문자를 제거)
with open_csv(TRAIN_ZIP) as f:
    head = pd.read_csv(f, nrows=5, encoding="utf-8-sig")
head

In [ ]:
# 샘플 1만 행으로 컬럼별 자료형(dtype) 확인
# object = 문자열, int64 = 정수
with open_csv(TRAIN_ZIP) as f:
    sample = pd.read_csv(f, nrows=10_000, encoding="utf-8-sig")
sample.dtypes

## 1-3. 전체 행을 나눠 읽으며 집계하기

파일 원본이 약 100MB이므로 10만 행씩 나눠 읽고(`chunksize`) 숫자만 모은다.
원본 행 전체를 메모리나 화면에 남기지 않는다.

In [ ]:
# 범주형으로 보이는 컬럼 (고유값 개수를 세어 볼 대상)
CAT_COLS = ["customer_type", "age", "gender", "region", "income_bracket",
            "occupation_group", "risk_grade", "credit_rating", "marital_status",
            "investment_propensity", "cross_coverage", "disease_history",
            "transaction_type", "transaction_keyword"]
# 숫자형 컬럼 (최솟값/최댓값을 볼 대상)
NUM_COLS = ["customer_seq", "product_amt", "product_period", "transaction_amt",
            "transaction_start_year", "transaction_last_year"]

def profile(zp):
    """CSV를 10만 행씩 읽으며 행 수, 결측, '-' 개수, 값 분포, 숫자 범위를 누적한다."""
    n_rows = 0
    null_cnt = None
    dash_cnt = None
    value_counts = {c: pd.Series(dtype="int64") for c in CAT_COLS}
    num_min, num_max = {}, {}
    ids = set()                      # 고유 고객 수를 세기 위한 집합
    rows_per_id = pd.Series(dtype="int64")
    with open_csv(zp) as f:
        for chunk in pd.read_csv(f, chunksize=100_000, encoding="utf-8-sig"):
            n_rows += len(chunk)
            # 컬럼별 결측(NaN) 개수 누적
            nc = chunk.isna().sum()
            null_cnt = nc if null_cnt is None else null_cnt + nc
            # 컬럼별 '-' 값 개수 누적 (결측을 '-'로 표시했을 가능성 확인용)
            dc = (chunk.astype(str) == "-").sum()
            dash_cnt = dc if dash_cnt is None else dash_cnt + dc
            # 범주형 값 분포 누적
            for c in CAT_COLS:
                value_counts[c] = value_counts[c].add(
                    chunk[c].value_counts(dropna=False), fill_value=0)
            # 숫자형 최솟값/최댓값 갱신
            for c in NUM_COLS:
                s = pd.to_numeric(chunk[c], errors="coerce")
                num_min[c] = min(num_min.get(c, float("inf")), s.min())
                num_max[c] = max(num_max.get(c, float("-inf")), s.max())
            # 고객 ID 수집 + 고객별 행 수 누적
            ids.update(chunk["customer_id"])
            rows_per_id = rows_per_id.add(chunk["customer_id"].value_counts(), fill_value=0)
    return dict(n_rows=n_rows, null_cnt=null_cnt, dash_cnt=dash_cnt,
                value_counts=value_counts, num_min=num_min, num_max=num_max,
                ids=ids, rows_per_id=rows_per_id.astype(int))

train = profile(TRAIN_ZIP)
print("Training 행 수:", train["n_rows"])
print("Training 고유 customer_id 수:", len(train["ids"]))

## 1-4. 질문 1·2: 한 행 = 고객 한 명인가? `customer_seq`의 의미는?

In [ ]:
# 고객 한 명당 몇 개의 행이 있는지 분포 확인
# (모두 1이면 한 행 = 고객 한 명, 2 이상이 있으면 한 고객이 여러 행)
train["rows_per_id"].value_counts().sort_index().rename("고객 수").to_frame()

In [ ]:
# customer_seq 값의 분포 (customer_id와 customer_seq 두 컬럼만 읽는다)
with open_csv(TRAIN_ZIP) as f:
    seq = pd.read_csv(f, usecols=["customer_id", "customer_seq"], encoding="utf-8-sig")
print(seq["customer_seq"].value_counts().sort_index().head(20))

In [ ]:
# 한 고객 안에서 customer_seq가 1, 2, 3...으로 이어지는지 확인
# 고객별로 (행 수)와 (customer_seq 최댓값)이 같으면 '고객 내 순번'일 가능성이 높다
g = seq.groupby("customer_id")["customer_seq"].agg(["count", "max", "min"])
print("행 수 == seq 최댓값 인 고객 비율:", (g["count"] == g["max"]).mean().round(4))
print("seq 최솟값 분포:")
print(g["min"].value_counts().head())
del seq, g  # 메모리 정리

## 1-5. 질문 3: 결측값과 `-` 값

In [ ]:
# 컬럼별 결측(NaN) 개수와 '-' 개수를 한 표로 비교
miss = pd.DataFrame({
    "결측(NaN)": train["null_cnt"],
    "'-' 개수": train["dash_cnt"],
})
miss["'-' 비율(%)"] = (miss["'-' 개수"] / train["n_rows"] * 100).round(2)
miss[(miss["결측(NaN)"] > 0) | (miss["'-' 개수"] > 0)]

In [ ]:
# '-'가 고객 유형(보험/증권)에 따라 달라지는지 확인
# 예: 보험 고객만 investment_propensity가 '-'라면 '해당 없음'일 가능성
with open_csv(TRAIN_ZIP) as f:
    t = pd.read_csv(f, usecols=["customer_type", "investment_propensity",
                                "cross_coverage", "disease_history"],
                    encoding="utf-8-sig")
for c in ["investment_propensity", "cross_coverage", "disease_history"]:
    print(f"--- {c} ---")
    print(pd.crosstab(t["customer_type"], t[c] == "-", margins=True)
            .rename(columns={True: "'-'", False: "값 있음"}))
del t

## 1-6. 범주형 컬럼 값 분포

In [ ]:
# 범주형 컬럼마다 고유값 개수와 상위 값 확인
for c in CAT_COLS:
    vc = train["value_counts"][c].sort_values(ascending=False).astype(int)
    print(f"\n=== {c} (고유값 {len(vc)}개) ===")
    print(vc.head(15).to_string())

## 1-7. 질문 4: 숫자형 컬럼 범위와 `product_period`

In [ ]:
# 숫자형 컬럼의 최솟값/최댓값
pd.DataFrame({"최솟값": train["num_min"], "최댓값": train["num_max"]})

In [ ]:
# product_period가 '연도'인지 확인: transaction_start_year와 비교
# product_period - transaction_start_year 가 상식적인 계약 기간(년)처럼 보이면 '만기 연도'일 가능성
with open_csv(TRAIN_ZIP) as f:
    p = pd.read_csv(f, usecols=["customer_type", "product_period",
                                "transaction_start_year"], encoding="utf-8-sig")
p["period_num"] = pd.to_numeric(p["product_period"], errors="coerce")
print("숫자로 변환 안 되는 product_period 값 예시:",
      p.loc[p["period_num"].isna(), "product_period"].unique()[:10])
p["diff"] = p["period_num"] - p["transaction_start_year"]
print(p.groupby("customer_type")["diff"].describe())
del p

## 1-8. 질문 5: Training과 Validation의 관계

In [ ]:
# Validation도 같은 방식으로 집계
valid = profile(VALID_ZIP)
print("Validation 행 수:", valid["n_rows"])
print("Validation 고유 customer_id 수:", len(valid["ids"]))

# 두 데이터에 동시에 나오는 고객이 몇 명인지 확인
overlap = train["ids"] & valid["ids"]
print("겹치는 고객 수:", len(overlap))
print("Validation 고객 중 Training에도 있는 비율:", round(len(overlap) / len(valid["ids"]), 4))

In [ ]:
# 컬럼 구성이 같은지, 범주 분포가 비슷한지 간단 비교 (age 예시)
cmp = pd.DataFrame({
    "Training(%)": train["value_counts"]["age"] / train["n_rows"] * 100,
    "Validation(%)": valid["value_counts"]["age"] / valid["n_rows"] * 100,
}).round(2)
cmp

## 1단계 정리 (직접 작성)

위 결과를 보고 아래 질문에 대한 답을 적어 보세요. 작성 후 검토를 요청하면 됩니다.

1. 한 행은 무엇을 뜻하는가? (고객 / 고객×상품 / 거래 …)
2. `customer_seq`의 의미:
3. `-` 값은 결측인가, 해당 없음인가? 어떻게 처리할 것인가?
4. `product_period`의 의미와 처리 방법:
5. Training과 Validation을 어떻게 사용할 것인가?
